## Feature Engineering and Modeling

In [2]:
# Importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna


In [5]:
# Load the dataset
df = pd.read_csv('data/train.csv')

In [6]:
# Encoding the target variable
df["Irrigation_Need"] = df["Irrigation_Need"].map({"High": 2,"Medium": 1, "Low": 0})

# Removing the id column as it is not useful for modeling
df.drop("id", axis=1, inplace=True)

In [7]:
# Feature engineering: Creating new features based on existing ones
df['dryness_score'] = (
    -df['Soil_Moisture']
    + df['Temperature_C']
    + df['Wind_Speed_kmh']
    - df['Rainfall_mm']
)

df['low_moisture'] = (df['Soil_Moisture'] < 20.5).astype(int)

df['high_temperature'] = (df['Temperature_C'] > 30).astype(int)

df['high_wind'] = (df['Wind_Speed_kmh'] > 12.5).astype(int)

df['low_rainfall'] = (df['Rainfall_mm'] < 850).astype(int)

df['temp_x_wind'] = df['Temperature_C'] * df['Wind_Speed_kmh']
df['temp_x_sunlight'] = df['Temperature_C'] * df['Sunlight_Hours']
df['wind_x_sunlight'] = df['Wind_Speed_kmh'] * df['Sunlight_Hours']

df['active_growth_stage'] = df['Crop_Growth_Stage'].isin(["Flowering", "Vegetative"]).astype(int)
df['stress_count'] = (
    df['low_moisture'] + df['high_temperature'] + df['high_wind'] + df['low_rainfall'] + df['active_growth_stage']
)

In [8]:
# Splitting the dataset into features and target variable
X = df.drop("Irrigation_Need", axis=1)
y = df["Irrigation_Need"]

In [9]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [10]:
# Differentiating between numerical and categorical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

In [11]:
preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='passthrough'
)

In [ ]:
# Using GridSearchCV to find the best hyperparameters for XGBoost because it performed better than Optuna
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss')
param_grid = {
    'xgb__n_estimators': [100, 200, 300, 400, 500],
    'xgb__max_depth': [3, 5, 7, 9],
    'xgb__learning_rate': [0.01, 0.1, 0.2],
    'xgb__subsample': [0.8, 1.0],
    'xgb__colsample_bytree': [0.8, 1.0]
}
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', xgb)
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best Hyperparameters:", grid_search.best_params_)
best_xgb = grid_search.best_estimator_